### Import libraries

In [1]:
import os
import pickle
from methyldl.deconvolution.xgbdeconvolver import *
from tqdm import tqdm
from collections import defaultdict
import torch.nn as nn
from copy import deepcopy
from methyldl.deconvolution.deep_deconvolvers.training import train_matrix_deconvolver
import os.path as Path
from torch.utils.data import DataLoader, TensorDataset
from methyldl.deconvolution.least_squares_deconvolvers import NNLSDeconvolver, PSLSDeconvolver
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

### Configuring target model and extracting computed average scores

In [2]:
from edautils import *
reads_data_path = '/home/luna.kuleuven.be/u0169940/Data/Loyfer/SoftLabelsForRRBSsplits_205files_pooled_Jaccard_hg38_mincpg_4_minlen_10_d041/'
dmr_label_column = "dmr_ctype_label"
mincpg_pointer = reads_data_path.find("mincpg_")
# classifier_model_path = f"../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_{dmr_label_column}_{reads_data_path[mincpg_pointer:]}"
classifier_model_path = "/home/luna.kuleuven.be/u0169940/Repos/methyldl/Experiments/pseudobulk/LookupClassifier_SoftLabels_Ours/pseudobulk"
mincpg = int(reads_data_path[mincpg_pointer+7:mincpg_pointer+8])
features_encoding = "extracted_numpy"
n_cell_types = num_dmr_groups = 39
n_pred_classes = 39 if "soft_labels" in classifier_model_path else 40

In [3]:
features_file = [x for x in os.listdir(classifier_model_path) if "features_cutoff" in x][0]

In [4]:
df = np.load(Path.join(classifier_model_path, features_file))

In [5]:
features_test = df["features_test"]
features_train = df["features_train"]
features_valid = df["features_valid"]
target_proportions = df["proportions"]

### Defining deconvolvers

In [6]:
n_features = 156

In [7]:
swn = nn.Sequential(
    nn.Linear(n_features, 1024),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(1024, 39),
    nn.Softmax(dim=-1)
)

mlp = nn.Sequential(
    nn.Linear(n_features, 512),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.GELU(),
    nn.Dropout(0.2),
    nn.Linear(256, n_features),
    nn.GELU(),
    nn.Dropout(0.1),
    nn.Linear(n_features, 39),
    nn.Softmax(dim=-1)
)

xgb_config = XGBDeconvolverConfig(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=1,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=100,
    random_state=42,
)

xgb = XGBoostDeconvolver(
        config=xgb_config,
        output_transform='clip_normalize',
        n_dmr_groups = num_dmr_groups,
        n_pred_classes = n_pred_classes,
        n_cell_types = n_cell_types,
        with_reject_features = False,
        process_inputs = False
    )

nnls = NNLSDeconvolver()
psls = PSLSDeconvolver()

### Loading deconvolvers weights

In [8]:
device = "cuda"

In [9]:
swn.to(device=device)
swn.load_state_dict(torch.load(Path.join(classifier_model_path, "swn_best_deconvolver.pt"), weights_only=True))
swn.eval()
mlp.to(device=device)
mlp.load_state_dict(torch.load(Path.join(classifier_model_path, "mlp_best_deconvolver.pt"), weights_only=True))
mlp.eval()
xgb = xgb.load(Path.join(classifier_model_path, "xgb_deconvolver.joblib"))
nnls = nnls.load(Path.join(classifier_model_path, "nnls_deconvolver.joblib"))
psls = psls.load(Path.join(classifier_model_path, "psls_deconvolver.joblib"))

### Generating predictions

In [10]:
results = defaultdict(tuple)
model_tripples = [
    (swn,"swn", "nn"),
    (mlp, "mlp", "nn"),
    (xgb, "xgb", "xgb"),
    (nnls, "nnls", "ls"),
    (psls, "psls", "ls")
]

In [11]:
def infer_multiple_deconvolvers(features, target_proportions, model_tripples):
    test_loader = DataLoader(
        TensorDataset(
            torch.FloatTensor(features), torch.FloatTensor(target_proportions)
        ),
        batch_size=2000,
    )
    results = defaultdict(tuple)
    for model, model_name, model_type in model_tripples:
        all_preds = []
        all_targets = target_proportions
        if model_type=="nn":
            all_targets = []
            with torch.no_grad():
                for X, y in test_loader:
                    X, y = X.to(device), y.to(device)
                    pred = model(X)
                    all_preds.append(pred.cpu())
                    all_targets.append(y.cpu())
            # Compute all metrics
            all_preds = torch.cat(all_preds, dim=0).numpy()
            all_targets = torch.cat(all_targets, dim=0)
            all_targets = all_targets.numpy()
        elif model_type=="xgb":
            all_preds = model._predict_raw(features)
            all_preds = model._transform_output(all_preds)
        elif model_type=="ls":
            if "nnls" in model_name:
                all_preds,_,_ = model.predict(features, n_workers=1)
            elif "psls" in model_name:
                all_preds = model.predict(features, n_workers =2)
        else:
            pass
        metrics = compute_deconvolution_metrics(all_preds, all_targets)
        all_preds = np.round(all_preds, 4)

        results[model_name] = (all_targets, all_preds, metrics)
    
    return results

In [12]:
results_test = infer_multiple_deconvolvers(features=features_test, target_proportions=target_proportions, model_tripples=model_tripples)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:05<00:00,  7.97it/s]


In [13]:
import pandas as pd

def results_to_dataframe(results: dict, class_names: list = None) -> pd.DataFrame:
    """
    Convert the results dict into a DataFrame matching the supplementary table columns.

    Expected results structure:
        results[model_name] = (all_targets, all_preds, metrics_dict)

    model_name convention assumed (adjust parsing as needed):
        e.g. "hard_rej__dsimir__dirichlet__xgb__linear"
              labeling__classifier__clf_calib__deconvolver__dec_calib
    """
    rows = []
    for model_name, (targets, preds, metrics) in results.items():
        # --- parse model_name into table keys ---
        parts = model_name.split("__")
        if len(parts) == 5:
            labeling, classifier, clf_calib, deconvolver, dec_calib = parts
        elif len(parts) == 3:
            # UXM case: e.g. "uxm__uxm__linear"
            labeling, deconvolver, dec_calib = parts
            classifier = "---"
            clf_calib = "---"
        else:
            # fallback: store raw name, fill manually
            labeling = ""
            classifier = ""
            clf_calib = ""
            deconvolver = model_name
            dec_calib = ""

        worst_class = metrics["worst_class_name"]
        if class_names is not None and isinstance(worst_class, int):
            worst_class = class_names[worst_class]

        rows.append({
            # "Labeling": labeling,
            # "Classifier": classifier,
            # "Clf. Calib.": clf_calib,
            "Deconvolver": deconvolver,
            # "Dec. Calib.": dec_calib,
            "R2": 1.0 - metrics["mse"] / targets.var() if hasattr(targets, 'var') else None,
            "LoA": f"[{metrics['loa_lower']:.4f}, {metrics['loa_upper']:.4f}]",
            "LoA width": round(metrics["loa_width"], 4),
            "LoA (worst)": f"[{metrics['worst_class_loa_lower']:.4f}, {metrics['worst_class_loa_upper']:.4f}]",
            "LoA width (worst)": round(metrics["worst_class_loa_width"], 4),
            "Worst class": worst_class,
            "MSE": round(metrics["mse"], 6),
            "MAE": round(metrics["mae"], 6),
            "KL": round(metrics["kl"], 6),
            "Cosine Sim": round(metrics["cosine_sim"], 6),
        })

    df = pd.DataFrame(rows)

    # Sort to match table grouping order
    df = df.sort_values(
        by=["Deconvolver"]
    ).reset_index(drop=True)

    return df

In [15]:
results_to_dataframe(results_test).to_dict()

{'Deconvolver': {0: 'mlp', 1: 'nnls', 2: 'psls', 3: 'swn', 4: 'xgb'},
 'R2': {0: 0.9387781620025635,
  1: 0.9538470633608868,
  2: 0.9562065379505802,
  3: 0.9542235136032104,
  4: 0.8824231546828311},
 'LoA': {0: '[-0.0441, 0.0441]',
  1: '[-0.0383, 0.0383]',
  2: '[-0.0373, 0.0373]',
  3: '[-0.0382, 0.0382]',
  4: '[-0.0612, 0.0612]'},
 'LoA width': {0: 0.0883, 1: 0.0766, 2: 0.0746, 3: 0.0763, 4: 0.1223},
 'LoA (worst)': {0: '[-0.1260, 0.1752]',
  1: '[-0.1099, 0.1281]',
  2: '[-0.1082, 0.1243]',
  3: '[-0.1069, 0.1443]',
  4: '[-0.1677, 0.1536]'},
 'LoA width (worst)': {0: 0.3012, 1: 0.2381, 2: 0.2325, 3: 0.2512, 4: 0.3214},
 'Worst class': {0: 11, 1: 11, 2: 11, 3: 11, 4: 11},
 'MSE': {0: 0.000507, 1: 0.000382, 2: 0.000363, 3: 0.000379, 4: 0.000974},
 'MAE': {0: 0.006313, 1: 0.005961, 2: 0.005781, 3: 0.005263, 4: 0.007957},
 'KL': {0: 0.129678, 1: 0.136793, 2: 0.132956, 3: 0.104392, 4: 0.211045},
 'Cosine Sim': {0: 0.968847,
  1: 0.982291,
  2: 0.982605,
  3: 0.977416,
  4: 0.950893

### Infering linear callibrators

In [ ]:
results_test.keys()

In [ ]:
def apply_fitted_callibration(results_test):
    results = defaultdict(tuple)
    for model_name in results_test.keys():
        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.load_calibration_parameters(Path.join(classifier_model_path, f"{model_name}_linear_calibrator.npz"))
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

    return results

In [ ]:
results_calibrated = apply_fitted_callibration(results_test)

In [ ]:
results_to_dataframe(results_calibrated)

### Fitting linear callibrators

In [16]:
results_valid = infer_multiple_deconvolvers(features=features_valid, target_proportions=target_proportions, model_tripples=model_tripples)

Predicting with PSLS in parallel: 100%|██████████| 1000/1000 [02:06<00:00,  7.89it/s]


In [17]:
results_to_dataframe(results_valid)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.946765,"[-0.0412, 0.0412]",0.0823,"[-0.0987, 0.1177]",0.2165,11,0.000441,0.006403,0.130958,0.973226
1,nnls,0.954884,"[-0.0379, 0.0379]",0.0758,"[-0.1069, 0.0994]",0.2062,11,0.000374,0.005912,0.138070,0.982478
2,psls,0.958740,"[-0.0362, 0.0362]",0.0725,"[-0.1044, 0.0913]",0.1958,11,0.000342,0.005520,0.128906,0.983211
3,swn,0.955743,"[-0.0375, 0.0375]",0.0750,"[-0.1088, 0.0993]",0.2082,0,0.000366,0.005377,0.106987,0.980348
4,xgb,0.884024,"[-0.0607, 0.0607]",0.1215,"[-0.1716, 0.1435]",0.3151,11,0.000960,0.008083,0.211100,0.952623


In [18]:
def apply_callibration(results_valid, results_test, save_calibrators = False):
    results = defaultdict(tuple)
    for model_name in results_valid.keys():
        valid_preds = results_valid[model_name][0]
        val_target = results_valid[model_name][1]

        test_preds = results_test[model_name][0]
        test_target = results_test[model_name][1]

        calibrator = LinearCalibrator()
        calibrator.fit(valid_preds, val_target)
        calibrated_test_pred, _ = calibrator.predict(test_preds)

        metrics = compute_deconvolution_metrics(calibrated_test_pred, test_target)
        calibrated_test_pred = np.round(calibrated_test_pred, 4)

        results[model_name] = (test_target, calibrated_test_pred, metrics)

        if save_calibrators:
            calibrator.save_calibration_parameters(f"{classifier_model_path}/{model_name}_linear_calibrator.npz")
    
    return results

In [19]:
calibrated_test_results = apply_callibration(results_valid, results_test, save_calibrators=True)

In [ ]:
results_to_dataframe(calibrated_test_results)

,Deconvolver,R2,LoA,LoA width,LoA (worst),LoA width (worst),Worst class,MSE,MAE,KL,Cosine Sim
0,mlp,0.948483,"[-0.0379, 0.0379]",0.0758,"[-0.1529, 0.1265]",0.2794,11,0.000374,0.006642,0.176790,0.974033
1,nnls,0.970284,"[-0.0274, 0.0274]",0.0548,"[-0.1060, 0.0846]",0.1905,11,0.000196,0.005710,0.163147,0.987151
2,psls,0.971699,"[-0.0270, 0.0270]",0.0540,"[-0.1031, 0.0786]",0.1817,11,0.000190,0.005480,0.168035,0.987800
3,swn,0.966848,"[-0.0305, 0.0305]",0.0611,"[-0.1270, 0.1012]",0.2282,11,0.000243,0.005213,0.128694,0.983601
4,xgb,0.943190,"[-0.0375, 0.0375]",0.0750,"[-0.0618, 0.0755]",0.1373,13,0.000366,0.008439,0.206533,0.978306


### Latex entry generation

In [ ]:
from collections import OrderedDict


def build_supp_table(
    entries,
    caption="Supplementary: Full results.",
    label="tab:supp_results",
):
    """
    Build a LaTeX supplementary table from a flat list of (classifier, calibration, data)
    tuples, where `data` is a dict-of-dicts in the shape returned by
    `pd.DataFrame.to_dict()` — i.e. {column_name: {row_idx: value, ...}, ...}.

    Each (classifier, calibration) pair contributes one row per Deconvolver in `data`.
    Rows are grouped by classifier via \\multirow; within a classifier, rows are grouped
    by Deconvolver so that calibration variants sit under the same Deconvolver header
    with \\cmidrule separators between different Deconvolvers.

    Columns emitted: Classifier | Deconvolver | Dec. Calib. | R^2 | LoA | LoA (worst) | MAE | MSE | KL
    No automatic highlighting is applied.

    Parameters
    ----------
    entries : list[tuple[str, str, dict]]
        Each tuple is (classifier_name, calibration_name, data_dict).
        `data_dict` must contain the keys:
            'Deconvolver', 'R2', 'LoA', 'LoA (worst)', 'MAE', 'MSE', 'KL'
    caption : str
    label : str

    Returns
    -------
    str : The full LaTeX table as a string.
    """
    # Deconvolver display names (upper-case) — input may use any casing
    decon_display = {
        'xgb': 'XGB', 'mlp': 'MLP', 'swn': 'SWN',
        'nnls': 'NNLS', 'psls': 'PSLS',
    }

    # 1) Group entries by classifier, preserving input order
    #    by_classifier[clf] = list of (calibration, data_dict)
    by_classifier = OrderedDict()
    for clf, calib, data in entries:
        by_classifier.setdefault(clf, []).append((calib, data))

    # 2) For each classifier, reorganize into {deconvolver: [(calib, row_values), ...]}
    #    preserving the Deconvolver order from the first calibration's data dict.
    def format_row_values(data, row_idx):
        r2 = data['R2'][row_idx] * 100  # convert to percentage like the example
        return {
            'R2': f"{r2:.2f}",
            'LoA': data['LoA'][row_idx],
            'LoA (worst)': data['LoA (worst)'][row_idx],
            'MAE': f"{data['MAE'][row_idx]:.6f}",
            'MSE': f"{data['MSE'][row_idx]:.6f}",
            'KL': f"{data['KL'][row_idx]:.6f}",
        }

    def collect_by_decon(calib_data_list):
        """Return OrderedDict: decon_name -> list of (calib_name, formatted_row_dict)."""
        by_decon = OrderedDict()
        for calib, data in calib_data_list:
            # Iterate row indices in the order they appear in the Deconvolver field
            for row_idx, decon_raw in data['Deconvolver'].items():
                decon = decon_display.get(decon_raw.lower(), decon_raw.upper())
                by_decon.setdefault(decon, []).append(
                    (calib, format_row_values(data, row_idx))
                )
        return by_decon

    # 3) Emit LaTeX
    lines = []
    lines.append(r"\begin{table}[H]")
    lines.append(r"  \centering")
    lines.append(rf"  \caption{{{caption}}}")
    lines.append(rf"  \label{{{label}}}")
    lines.append(r"  {\footnotesize")
    lines.append(r"    \resizebox{\textwidth}{!}{%")
    lines.append(r"      \begin{tabular}{l cc cccc cc}")
    lines.append(r"        \toprule")
    lines.append(
        r"        \textbf{Classifier} & \textbf{Deconvolver} & \textbf{Dec.\ Calib.}"
        r" & \textbf{$R^2$} & \textbf{LoA} & \textbf{LoA (worst)} & \textbf{MAE}"
        r" & \textbf{MSE} & \textbf{KL} \\"
    )
    lines.append(r"        \midrule")

    classifier_items = list(by_classifier.items())
    for clf_i, (clf, calib_data_list) in enumerate(classifier_items):
        by_decon = collect_by_decon(calib_data_list)

        # Total row count for this classifier = sum of rows across all its deconvolvers
        total_rows_for_clf = sum(len(v) for v in by_decon.values())
        lines.append(rf"        % --- {clf} ---")
        lines.append(rf"        \multirow{{{total_rows_for_clf}}}{{*}}{{{clf}}}")

        decon_items = list(by_decon.items())
        for dec_i, (decon, calib_rows) in enumerate(decon_items):
            n_calib = len(calib_rows)

            for row_j, (calib, vals) in enumerate(calib_rows):
                if row_j == 0 and n_calib > 1:
                    decon_cell = rf"\multirow{{{n_calib}}}{{*}}{{{decon}}}"
                elif row_j == 0 and n_calib == 1:
                    decon_cell = decon
                else:
                    decon_cell = ""  # continuation row under the multirow

                # Build the full row
                row = (
                    f"                            & {decon_cell:<20} & {calib:<20} "
                    f"& {vals['R2']:<8} & {vals['LoA']:<25} & {vals['LoA (worst)']:<25} "
                    f"& {vals['MAE']:<10} & {vals['MSE']:<10} & {vals['KL']:<10} \\\\"
                )
                lines.append(row)

            # cmidrule between deconvolvers (but not after the last deconvolver
            # of the last classifier, since \bottomrule follows)
            is_last_decon = (dec_i == len(decon_items) - 1)
            is_last_clf = (clf_i == len(classifier_items) - 1)
            if not is_last_decon:
                lines.append(r"        \cmidrule(l){2-9}")
            elif not is_last_clf:
                # end of a classifier block, but another classifier follows
                lines.append(r"        \midrule")

        # (no separator after the very last classifier's last deconvolver;
        # \bottomrule handles it)

    lines.append(r"        \bottomrule")
    lines.append(r"      \end{tabular}%")
    lines.append(r"    }}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

In [26]:
custom_order = {'xgb': 0, 'mlp': 1, 'swn': 2,'nnls':3, 'psls':4} 

In [27]:
entries = [
    ('Lookup Classifier',  'None',   results_to_dataframe(results_test).sort_values(by=['Deconvolver'], key=lambda x: x.map(custom_order)).to_dict()),
    ('Lookup Classifier',  'Linear', results_to_dataframe(calibrated_test_results).sort_values(by=['Deconvolver'], key=lambda x: x.map(custom_order)).to_dict()),
]

In [28]:
print(build_supp_table(entries))

\begin{table}[H]
  \centering
  \caption{Supplementary: Full results.}
  \label{tab:supp_results}
  {\footnotesize
    \resizebox{\textwidth}{!}{%
      \begin{tabular}{l cc cccc cc}
        \toprule
        \textbf{Classifier} & \textbf{Deconvolver} & \textbf{Dec.\ Calib.} & \textbf{$R^2$} & \textbf{LoA} & \textbf{LoA (worst)} & \textbf{MAE} & \textbf{MSE} & \textbf{KL} \\
        \midrule
        % --- Lookup Classifier ---
        \multirow{10}{*}{Lookup Classifier}
                            & \multirow{2}{*}{XGB} & None                 & 88.24    & [-0.0612, 0.0612]         & [-0.1677, 0.1536]         & 0.007957   & 0.000974   & 0.211045   \\
                            &                      & Linear               & 94.32    & [-0.0375, 0.0375]         & [-0.0618, 0.0755]         & 0.008439   & 0.000366   & 0.206533   \\
        \cmidrule(l){2-9}
                            & \multirow{2}{*}{MLP} & None                 & 93.88    & [-0.0441, 0.0441]         & [-0.1260, 0.1752]  